# ScriptChunker — Pre-Embedding Script Segmentation

Parses a tagged script file into **scene-level** and **sentence-level** chunks, ready to be embedded and indexed into the FEDE vector store.

- `SceneChunk` — one record per scene, aggregating all lines within that scene.
- `SentenceChunk` — one record per individual `N:` / `D:` / `T:` line.

After embeddings are computed, call `to_scene_records` / `to_sentence_records` to get objects compatible with `ScriptIndexer.index_movie_batch`.

In [1]:
import sys, os
# Make the repo root importable when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

In [ ]:
import numpy as np

from collections import Counter

from utils.chunker import ScriptChunker, SceneChunk, SentenceChunk
from vector_db.indexer import SceneRecord, SentenceRecord, ScriptIndexer, QdrantConfig
from vector_db import initialize_all_collections, ScriptRetriever

## Parse 8MM

In [3]:
chunker = ScriptChunker(
    movie_name="8MM",
    tagged_path="../data/scripts/parsed/tagged/8MM_parsed.txt",
)
scene_chunks, sentence_chunks = chunker.parse()
print(f"Parsed {len(scene_chunks)} scenes and {len(sentence_chunks)} sentences.")

Parsed 282 scenes and 1342 sentences.


In [4]:
type_counts = Counter(s.line_type for s in sentence_chunks)
print(f"Total scenes   : {len(scene_chunks)}")
print(f"Total sentences: {len(sentence_chunks)}")
print(f"By line_type   : {dict(type_counts)}")

Total scenes   : 282
Total sentences: 1342
By line_type   : {'description': 539, 'dialogue': 790, 'transition': 13}


## Example SceneChunks

In [6]:
for sc in scene_chunks[:3]:
    print(f"--- {sc.scene_id} ---")
    print(f"  title     : {sc.scene_title}")
    print(f"  characters: {sc.character_names}")
    preview = sc.text[:200].replace('\n', ' | ')
    print(f"  text      : {sc.text}...")
    print()

--- scene_0000 ---
  title     : INT. MIAMI AIRPORT, TERMINAL -- DAY
  characters: []
  text      : INT. MIAMI AIRPORT, TERMINAL -- DAY
Amongst the weary tourist families and solitary businessmen sits TOM WELLES, middle-aged, hair neat, suit crisp and gray. He's eating crackers from a cellophane package, sipping soda from a paper cup, watching an ARRIVAL GATE. AT THE GATE PASSENGERS arrive: the paunchy, graying men of First Class leading the pack, except for a handsome YOUNG REPUBLICAN poster boy hurrying along. ACROSS THE TERMINAL Welles gets up and FOLLOWS......

--- scene_0001 ---
  title     : EXT. MIAMI AIRPORT, CURBSIDE -- DAY
  characters: []
  text      : EXT. MIAMI AIRPORT, CURBSIDE -- DAY
Welles comes outside, squinting in the sun, moving down the sidewalk, looking back over his shoulder... The Young Republican is lead to a waiting LIMO by a DRIVER. Welles moves to the nearby TAXI STAND......

--- scene_0002 ---
  title     : INT. TAXI -- DAY
  characters: ['CAB DRIVER', 'WEL

## Example SentenceChunks by type

In [8]:
dialogues    = [s for s in sentence_chunks if s.line_type == "dialogue"]
descriptions = [s for s in sentence_chunks if s.line_type == "description"]
transitions  = [s for s in sentence_chunks if s.line_type == "transition"]

print("=== Dialogue (first 5) ===")
for s in dialogues[:5]:
    print(f"  [{s.scene_id}] {s.character_name}: {s.text}")

print("\n=== Description (first 5) ===")
for s in descriptions[:5]:
    print(f"  [{s.scene_id}] {s.text}")

print("\n=== Transitions (all) ===")
for s in transitions:
    print(f"  [{s.scene_id}] {s.text}")

=== Dialogue (first 5) ===
  [scene_0002] CAB DRIVER: Where to?
  [scene_0002] WELLES: Follow that limousine. Don't get too close, don't let it get too far away. Just keep with it.
  [scene_0002] CAB DRIVER: You kidding?
  [scene_0002] WELLES: Nope.
  [scene_0002] CAB DRIVER: Uh, listen... you're not supposed to be smoking in here. I'm sorry, that's company policy...

=== Description (first 5) ===
  [scene_0000] Amongst the weary tourist families and solitary businessmen sits TOM WELLES, middle-aged, hair neat, suit crisp and gray. He's eating crackers from a cellophane package, sipping soda from a paper cup, watching an ARRIVAL GATE. AT THE GATE PASSENGERS arrive: the paunchy, graying men of First Class leading the pack, except for a handsome YOUNG REPUBLICAN poster boy hurrying along. ACROSS THE TERMINAL Welles gets up and FOLLOWS...
  [scene_0001] Welles comes outside, squinting in the sun, moving down the sidewalk, looking back over his shoulder... The Young Republican is lead to a

## Convert to Records (placeholder embeddings)

In [ ]:
VECTOR_SIZE = 768
zero_emb = [0.0] * VECTOR_SIZE

scene_records    = chunker.to_scene_records([zero_emb] * len(scene_chunks))
sentence_records = chunker.to_sentence_records([zero_emb] * len(sentence_chunks))

print(f"scene_records   : {len(scene_records)}")
print(f"sentence_records: {len(sentence_records)}")
print()
print("First SceneRecord   :", scene_records[0])
print()
print("First SentenceRecord:", sentence_records[0])

scene_records   : 282
sentence_records: 1342

First SceneRecord   : SceneRecord(movie_id='8mm', movie_title='8MM', scene_id='scene_0000', scene_index=0, text="INT. MIAMI AIRPORT, TERMINAL -- DAY\nAmongst the weary tourist families and solitary businessmen sits TOM WELLES, middle-aged, hair neat, suit crisp and gray. He's eating crackers from a cellophane package, sipping soda from a paper cup, watching an ARRIVAL GATE. AT THE GATE PASSENGERS arrive: the paunchy, graying men of First Class leading the pack, except for a handsome YOUNG REPUBLICAN poster boy hurrying along. ACROSS THE TERMINAL Welles gets up and FOLLOWS...", embedding=[0.011154510080814362, -0.03806978091597557, 0.027471106499433517, 0.03443042188882828, -0.07141982764005661, -0.0476677380502224, 0.004679740872234106, -0.011576415039598942, -0.0006150251720100641, -0.03122662752866745, 0.03219134733080864, 0.028471944853663445, 0.002417127601802349, 0.04126392677426338, 0.0171137023717165, -0.03145536035299301, 0.01349853

## ScriptIndexer compatibility

In [26]:
sr = scene_records[0]
sent_r = sentence_records[0]

print("SceneRecord fields:")
print(f"  movie_id       : {sr.movie_id}")
print(f"  movie_title    : {sr.movie_title}")
print(f"  scene_id       : {sr.scene_id}")
print(f"  scene_index    : {sr.scene_index}")
print(f"  scene_title    : {sr.scene_title}")
print(f"  character_names: {sr.character_names}")
print(f"  embedding len  : {len(sr.embedding)}")
print()
print("SentenceRecord fields:")
print(f"  line_type          : {sent_r.line_type}")
print(f"  character_name     : {sent_r.character_name}")
print(f"  position_in_script : {sent_r.position_in_script}")
print()
print("isinstance checks:")
print(f"  scene_records[0]    is SceneRecord   : {isinstance(sr, SceneRecord)}")
print(f"  sentence_records[0] is SentenceRecord: {isinstance(sent_r, SentenceRecord)}")
print()
# To index into Qdrant once real embeddings are available:
# from vector_db.indexer import ScriptIndexer
# indexer = ScriptIndexer()
# indexer.index_movie_batch(scene_records, sentence_records)

SceneRecord fields:
  movie_id       : 8mm
  movie_title    : 8MM
  scene_id       : scene_0000
  scene_index    : 0
  scene_title    : INT. MIAMI AIRPORT, TERMINAL -- DAY
  character_names: []
  embedding len  : 768

SentenceRecord fields:
  line_type          : description
  character_name     : None
  position_in_script : 0

isinstance checks:
  scene_records[0]    is SceneRecord   : True
  sentence_records[0] is SentenceRecord: True



In [27]:
manager = initialize_all_collections()
print(manager.list_collections())  # ['scenes', 'sentences']

indexer = ScriptIndexer()
indexer.index_movie_batch(scene_records, sentence_records)

['scenes', 'sentences']


In [ ]:

rng = np.random.default_rng(42)
def unit_vector(size: int) -> list:
    v = rng.standard_normal(size).astype(np.float32)
    return (v / np.linalg.norm(v)).tolist()

query_vector = unit_vector(VECTOR_SIZE)
retriever = ScriptRetriever()

# --- Hierarchical search (recommended for production) ---
results = retriever.hierarchical_search(
    query_embedding=query_vector,
    top_k=20,
    sentence_pool=100,
)

for scene in results:
    print(f"[{scene.score:.4f}] {scene.movie_title} — {scene.scene_title}")
    print(f"  Characters: {scene.character_names}")
    print(f"  {scene.text[:120]}...")

# --- Flat scene search (for ablation / baseline comparison) ---
scene_results = retriever.search_scenes(query_embedding=query_vector, top_k=20)
print(scene_results[0])

# --- Flat sentence search ---
sent_results = retriever.search_sentences(query_embedding=query_vector, top_k=50)
print(sent_results[0])

# --- Filter to a single movie (evaluation use case) ---
filtered = retriever.search_scenes(
    query_embedding=query_vector,
    top_k=5,
    movie_id_filter="8mm",  # matches chunker.movie_id for movie_name="8MM"
)
print(filtered[0])

UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b''